# MNIST — Conv2d (4 filtros, padding=1)

**Propósito**: Confirmar que el bug de `Conv2d.backward()` con `padding > 0` es independiente de la cantidad de filtros.

**Arquitectura**: Conv2d(4 filtros, kernel 3×3, padding=1) → Dense(10), MSE loss.

**Resultado esperado**: El entrenamiento falla con `loss diverged: NaN or Inf detected`, igual que con 1 filtro.

**Bug**: El gradiente de los kernels se computa como `x_bc (28×28) corr_valid ud_bf (28×28) → (1×1)` en lugar de `(3×3)`. Con 4 filtros el volumen de parámetros es mayor pero el bug es exactamente el mismo — cada uno de los 9 pesos de cada kernel recibe el gradiente inflado ×9.

## 1. Setup & Descarga MNIST

In [8]:
import os, struct, gzip, shutil, urllib.request, subprocess, time, datetime
import numpy as np

NWORKERS = 3
NSERVERS = 2
BASE_WORKER_PORT = 50000
BASE_SERVER_PORT = 40000
WORKER_ADDRS = [f'worker-{i}:{BASE_WORKER_PORT + i}' for i in range(NWORKERS)]
SERVER_ADDRS = [f'server-{i}:{BASE_SERVER_PORT + i}' for i in range(NSERVERS)]

DOCKER_DIR   = os.path.abspath('../../docker')
COMPOSE_FILE = os.path.abspath('../../compose.yaml')

URLS = {
    'train_images': 'https://storage.googleapis.com/cvdf-datasets/mnist/train-images-idx3-ubyte.gz',
    'train_labels': 'https://storage.googleapis.com/cvdf-datasets/mnist/train-labels-idx1-ubyte.gz',
    'test_images':  'https://storage.googleapis.com/cvdf-datasets/mnist/t10k-images-idx3-ubyte.gz',
    'test_labels':  'https://storage.googleapis.com/cvdf-datasets/mnist/t10k-labels-idx1-ubyte.gz',
}

RAW_DIR          = 'data/mnist_raw'
TRAIN_SAMPLES_BIN = 'data/mnist_train_samples.bin'
TRAIN_LABELS_BIN  = 'data/mnist_train_labels.bin'
TEST_SAMPLES_BIN  = 'data/mnist_test_samples.bin'
TEST_LABELS_BIN   = 'data/mnist_test_labels.bin'
SAFETENSORS_PATH  = 'data/mnist_conv_4filters_model.safetensors'

X_SIZE = 784
Y_SIZE = 10

def download(name, url):
    os.makedirs(RAW_DIR, exist_ok=True)
    gz  = os.path.join(RAW_DIR, name + '.gz')
    raw = os.path.join(RAW_DIR, name)
    if os.path.exists(raw):
        print(f'  already exists: {raw}')
        return raw
    print(f'  downloading {name}...')
    urllib.request.urlretrieve(url, gz)
    with gzip.open(gz, 'rb') as fi, open(raw, 'wb') as fo:
        shutil.copyfileobj(fi, fo)
    os.remove(gz)
    return raw

def read_images(path):
    with open(path, 'rb') as f:
        _, n, rows, cols = struct.unpack('>IIII', f.read(16))
        raw = f.read(n * rows * cols)
    ppi = rows * cols
    return [[b / 255.0 for b in raw[i * ppi:(i + 1) * ppi]] for i in range(n)]

def read_labels(path):
    with open(path, 'rb') as f:
        _, n = struct.unpack('>II', f.read(8))
        return list(f.read(n))

def write_split_bins(images, labels, samples_path, labels_path):
    os.makedirs(os.path.dirname(samples_path), exist_ok=True)
    os.makedirs(os.path.dirname(labels_path),  exist_ok=True)
    with open(samples_path, 'wb') as fs, open(labels_path, 'wb') as fl:
        for img, lbl in zip(images, labels):
            one_hot = [0.0] * Y_SIZE
            one_hot[lbl] = 1.0
            fs.write(struct.pack(f'{X_SIZE}f', *img))
            fl.write(struct.pack(f'{Y_SIZE}f', *one_hot))
    print(f'  wrote {len(images)} samples -> {samples_path}')

for name, url in URLS.items():
    download(name, url)

if not (os.path.exists(TRAIN_SAMPLES_BIN) and os.path.exists(TRAIN_LABELS_BIN)):
    write_split_bins(
        read_images(os.path.join(RAW_DIR, 'train_images')),
        read_labels(os.path.join(RAW_DIR, 'train_labels')),
        TRAIN_SAMPLES_BIN, TRAIN_LABELS_BIN,
    )
else:
    print('training set already exists')

if not (os.path.exists(TEST_SAMPLES_BIN) and os.path.exists(TEST_LABELS_BIN)):
    write_split_bins(
        read_images(os.path.join(RAW_DIR, 'test_images')),
        read_labels(os.path.join(RAW_DIR, 'test_labels')),
        TEST_SAMPLES_BIN, TEST_LABELS_BIN,
    )
else:
    print('test set already exists')

  already exists: data/mnist_raw/train_images
  already exists: data/mnist_raw/train_labels
  already exists: data/mnist_raw/test_images
  already exists: data/mnist_raw/test_labels
training set already exists
test set already exists


## 2. Arquitectura — Conv2d 4 filtros

| Layer | Config | Output shape | Params |
|---|---|---|---|
| Conv2d | 4 filtros, kernel 3×3, stride=1, **padding=1** | (4, 28, 28) | 40 |
| Dense  | 3136 → 10, sin activación | (10,) | 31 370 |

Total: **31 410 parámetros**

> Con 4 filtros el output del conv tiene shape `(4, 28, 28)` = 3136 features antes del flatten.

In [9]:
CONV_LAYERS = [
    {'input_dim': (1, 28, 28), 'kernel_dim': (4, 1, 3), 'stride': 1, 'padding': 1, 'act': None},
]
DENSE_LAYERS = [
    {'size': 10, 'act': None},
]

FLAT_AFTER_CONV = 4 * 28 * 28  # 3136 — same padding keeps spatial dims

## 3. Start Workers & Servers

In [10]:
import getpass

sudo_password = getpass.getpass('sudo password: ')

subprocess.run(
    ['python3', f'{DOCKER_DIR}/gen_compose.py'],
    env={**os.environ, 'WORKERS': str(NWORKERS), 'SERVERS': str(NSERVERS), 'RELEASE': 'true'},
    check=True,
)

subprocess.run(
    ['sudo', '-S', '-E', 'python3', f'{DOCKER_DIR}/fill_hosts.py'],
    input=sudo_password + '\n',
    text=True,
    env={**os.environ, 'WORKERS': str(NWORKERS), 'SERVERS': str(NSERVERS)},
    check=True,
)

subprocess.run(
    ['docker', 'compose', '-f', COMPOSE_FILE, 'up', '--build', '-d', '--remove-orphans'],
    check=True,
)

time.sleep(3)
print('All entities running.')

[sudo] password for alejo:  Image distributed-training-worker-2 Building 
 Image distributed-training-server-0 Building 
 Image distributed-training-server-1 Building 
 Image distributed-training-worker-0 Building 
 Image distributed-training-worker-1 Building 


#1 [internal] load local bake definitions
#1 reading from stdin 2.70kB done
#1 DONE 0.0s

#2 [server-0 internal] load build definition from Dockerfile
#2 transferring dockerfile: 982B done
#2 DONE 0.0s

#3 [worker-0 internal] load build definition from Dockerfile
#3 transferring dockerfile: 1.03kB done
#3 DONE 0.0s

#4 [worker-2 internal] load metadata for docker.io/library/rust:1.92
#4 DONE 1.9s

#5 [worker-0 internal] load metadata for docker.io/library/debian:bookworm-slim
#5 DONE 1.9s

#6 [worker-1 internal] load .dockerignore
#6 transferring context: 172B done
#6 DONE 0.0s

#7 [worker-2 stage-3 1/2] FROM docker.io/library/debian:bookworm-slim@sha256:f9c6a2fd2ddbc23e336b6257a5245e31f996953ef06cd13a59fa0a1df2d5c252
#7 DONE 0.0s

#8 [worker-2 chef 1/3] FROM docker.io/library/rust:1.92@sha256:f58923369ba295ae1f60bc49d03f2c955a5c93a0b7d49acfb2b2a65bebaf350d
#8 DONE 0.0s

#9 [worker-0 internal] load build context
#9 transferring context: 2.42MB 1.4s done
#9 DONE 1.4s

#10 [server-1 chef

 Image distributed-training-worker-1 Built 
 Image distributed-training-worker-2 Built 
 Image distributed-training-server-0 Built 
 Image distributed-training-server-1 Built 
 Image distributed-training-worker-0 Built 
 Container server-0 Recreate 
 Container server-1 Recreate 
 Container server-0 Recreated 
 Container server-1 Recreated 
 Container worker-1 Recreate 
 Container worker-2 Recreate 
 Container worker-0 Recreate 
 Container worker-2 Recreated 
 Container worker-1 Recreated 
 Container worker-0 Recreated 
 Container server-0 Starting 
 Container server-1 Starting 
 Container server-1 Started 
 Container server-0 Started 
 Container worker-2 Starting 
 Container worker-0 Starting 
 Container worker-1 Starting 
 Container worker-0 Started 
 Container worker-2 Started 
 Container worker-1 Started 


All entities running.


## 4. Entrenamiento — esperamos fallo

Misma falla que con 1 filtro. El bug no depende del número de filtros sino del valor de `padding`.

In [11]:
import orchestra
from orchestra import Sequential, orchestrate
from orchestra.arch import Conv2d, Dense
from orchestra.initialization import Kaiming
from orchestra.datasets import LocalDataset
from orchestra.optimizers import GradientDescent
from orchestra.sync import BarrierSync
from orchestra.store import BlockingStore
from orchestra.loss_fns import Mse
from orchestra.serializer import SparseSerializer

model = Sequential([
    *[
        Conv2d(
            input_dim=l['input_dim'],
            kernel_dim=l['kernel_dim'],
            stride=l['stride'],
            padding=l['padding'],
            init=Kaiming(),
            act_fn=None,
        )
        for l in CONV_LAYERS
    ],
    *[
        Dense(l['size'], Kaiming(), None)
        for l in DENSE_LAYERS
    ],
])

training = orchestra.parameter_server(
    worker_addrs=WORKER_ADDRS,
    server_addrs=SERVER_ADDRS,
    dataset=LocalDataset(TRAIN_SAMPLES_BIN, TRAIN_LABELS_BIN, x_size=784, y_size=10),
    optimizer=GradientDescent(lr=0.01),
    loss_fn=Mse(),
    sync=BarrierSync(),
    store=BlockingStore(),
    max_epochs=50,
    batch_size=128,
    offline_epochs=2,
    seed=42,
    early_stopping_tolerance=1e-6,
)

training_success = False
trained = None

try:
    print('Starting training session...')
    session = orchestrate(model, training)
    trained = session.wait()
    training_success = True
    print(f'Training complete — {len(trained.weights())} parameters')
    trained.save_safetensors(SAFETENSORS_PATH)
except Exception as e:
    print(f'[ERROR] Training failed: {e}')
    print()
    print('=== BUG ANALYSIS ===')
    print('File:  machine_learning/src/arch/layers/conv2d.rs, fn backward()')
    print('Same bug as conv_1filter: padding=1 causes ud_bf to be (28x28).')
    print('With 4 filters: for each (b, f, c) triple, x_bc (28x28) corr_valid ud_bf (28x28)')
    print('  = (1x1) broadcast to (3x3) -> 4 kernels * 9 positions each, all wrong.')
    print('Conclusion: the bug is independent of filter count.')

Starting training session...
  epoch 3/50 avg_loss=0.07706661
  epoch 3/50 avg_loss=0.07738331
  epoch 3/50 avg_loss=0.07733919
  epoch 6/50 avg_loss=0.06913996
  epoch 6/50 avg_loss=0.06095802
  epoch 6/50 avg_loss=0.05249428
  epoch 9/50 avg_loss=0.05233707
  epoch 9/50 avg_loss=0.05215351
  epoch 9/50 avg_loss=0.05198741
  epoch 12/50 avg_loss=0.05029761
  epoch 12/50 avg_loss=0.04862800
  epoch 12/50 avg_loss=0.04686875
  epoch 15/50 avg_loss=0.04981947
  epoch 15/50 avg_loss=0.05277209
  epoch 15/50 avg_loss=0.05578242
  epoch 18/50 avg_loss=0.05492776
  epoch 18/50 avg_loss=0.05408199
  epoch 18/50 avg_loss=0.05320629
  epoch 21/50 avg_loss=0.05175640
  epoch 21/50 avg_loss=0.05031635
  epoch 21/50 avg_loss=0.04892131
  epoch 24/50 avg_loss=0.04844140
  epoch 24/50 avg_loss=0.04797522
  epoch 24/50 avg_loss=0.04746484
  epoch 27/50 avg_loss=0.04776300
  epoch 27/50 avg_loss=0.04808955
  epoch 27/50 avg_loss=0.04845760
  epoch 30/50 avg_loss=0.04750988
  epoch 30/50 avg_loss=0.046

## 5. Logs de Docker → /tmp/

In [12]:
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
LOG_PATH = f'/tmp/mnist_conv_4filters_{timestamp}.log'

with open(LOG_PATH, 'w') as f:
    f.write(f'=== MNIST Conv 4 filters — Docker Logs ===\n')
    f.write(f'=== Collected at {datetime.datetime.now().isoformat()} ===\n\n')
    f.write(f'=== training_success={training_success} ===\n\n')
    for name in [f'server-{i}' for i in range(NSERVERS)] + [f'worker-{i}' for i in range(NWORKERS)]:
        result = subprocess.run(
            ['docker', 'logs', '--timestamps', name],
            capture_output=True, text=True,
        )
        f.write(f'\n{"="*60}\n=== {name} ===\n{"="*60}\n')
        f.write(result.stdout)
        if result.stderr:
            f.write(result.stderr)

print(f'Docker logs saved to: {LOG_PATH}')

Docker logs saved to: /tmp/mnist_conv_4filters_20260428_043148.log


## 6. Stop Workers & Servers

In [13]:
print('Stopping all entities...')
subprocess.run(
    ['docker', 'compose', '-f', COMPOSE_FILE, 'down'],
    check=True,
)
print('Done.')

Stopping all entities...


 Container worker-2 Stopping 
 Container worker-0 Stopping 
 Container worker-1 Stopping 
 Container worker-2 Stopped 
 Container worker-2 Removing 
 Container worker-0 Stopped 
 Container worker-0 Removing 
 Container worker-1 Stopped 
 Container worker-1 Removing 
 Container worker-1 Removed 
 Container worker-2 Removed 
 Container worker-0 Removed 
 Container server-1 Stopping 
 Container server-0 Stopping 


Done.


 Container server-1 Stopped 
 Container server-1 Removing 
 Container server-1 Removed 
 Container server-0 Stopped 
 Container server-0 Removing 
 Container server-0 Removed 
 Network distributed-training_training-network Removing 
 Network distributed-training_training-network Removed 
